# Faz 2 — Türkçe AYM Kararı Sınıflandırıcısı (cased BERTurk)

**Proje:** Yüksek-riskli Türkçe metin sınıflandırmasında XAI faithfulness karşılaştırması.

**Bu notebook:** `dbmdz/bert-base-turkish-128k-cased` modelini `icgcihan/Turkish_Constutional_Court_Decisions`
üzerinde ihlal/ihlal-değil ikili sınıflandırma için ince ayarlar; Faz 3 (XAI) için modeli + örnek havuzunu kaydeder.

### Çalıştırmadan önce (Kaggle sağ panel):
1. **Settings → Accelerator → GPU (T4 x1 veya P100)**
2. **Settings → Internet → ON** (veri seti + model indirilecek)
3. Run All. Süre ~30–60 dk.

Çıktılar `/kaggle/working/` altında; bitince **Output** sekmesinden indir:
`model/` (Faz 3 için), `test_predictions.csv`, `example_pool.json`, `metrics.json`.

In [ ]:
# 1) Ortam (Kaggle'da çoğu kurulu; sürümleri hizalıyoruz)
!pip -q install -U "transformers>=4.44" "datasets>=2.20" scikit-learn 2>/dev/null
import transformers, datasets, torch
print("transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 2) Konfig + tekrarlanabilirlik
import numpy as np, random, os, json
from transformers import set_seed
SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

DATASET   = "icgcihan/Turkish_Constutional_Court_Decisions"
MODEL     = "dbmdz/bert-base-turkish-128k-cased"
TEXT_COL  = "text"
LABEL_COL = "labels"
MAX_LEN   = 512          # head truncation (D1)
EPOCHS    = 3
LR        = 2e-5
BATCH     = 8            # 128k vocab + 512 len için güvenli; grad_accum ile efektif 16
GRAD_ACC  = 2
OUTDIR    = "/kaggle/working"


In [ ]:
# 3) Veri
from datasets import load_dataset
ds = load_dataset(DATASET)
print({k: len(ds[k]) for k in ds})
from collections import Counter
for k in ds:
    print(k, dict(Counter(ds[k][LABEL_COL])))
# etiketler zaten 0/1 int; sınıf adları (polarite Faz 2 sonrası teyit edilecek)
NUM_LABELS = 2

In [ ]:
# 4) Tokenizasyon (head-512, cased → diacritic korunur)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.truncation_side == "right", "head truncation için truncation_side='right' olmalı"

def prep(batch):
    enc = tok(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

keep = ds["train"].column_names
ds_tok = ds.map(prep, batched=True, remove_columns=keep)
print(ds_tok["train"][0].keys())

In [ ]:
# 5) Model + metrikler
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import f1_score, precision_recall_fscore_support, average_precision_score, confusion_matrix
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)
collator = DataCollatorWithPadding(tok)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits); preds = probs.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0,1], zero_division=0)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_pos": f1[1], "f1_neg": f1[0],
        "prec_pos": p[1], "rec_pos": r[1],
        "pr_auc_pos": average_precision_score(labels, probs[:,1]),
    }

In [ ]:
# 6) Eğitim (sürüm-uyumlu TrainingArguments)
import inspect
from transformers import TrainingArguments, Trainer

_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
_eval_key = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
args_kw = dict(
    output_dir=f"{OUTDIR}/ckpt",
    learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=50, save_total_limit=1, seed=SEED,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    report_to="none",
)
args_kw[_eval_key] = "epoch"; args_kw["save_strategy"] = "epoch"
args = TrainingArguments(**args_kw)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
                  processing_class=tok, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# 7) Test değerlendirmesi + sızıntı kontrolü
test_out = trainer.predict(ds_tok["test"])
metrics = {k.replace("test_",""): float(v) for k,v in test_out.metrics.items() if isinstance(v,(int,float))}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

probs = softmax(test_out.predictions); preds = probs.argmax(-1); labels = np.array(test_out.label_ids)
print("Confusion matrix [0,1]:\n", confusion_matrix(labels, preds, labels=[0,1]))

if metrics.get("test_f1_macro", metrics.get("f1_macro",0)) > 0.97:
    print("\n⚠️ UYARI: F1 çok yüksek — olası etiket sızıntısı. Metinlerde AYM hükmü kalmış olabilir, incele!")
else:
    print("\n✅ F1 makul bantta — sızıntı-güvenli 'olaylardan tahmin' görevi ile tutarlı.")

In [ ]:
# 8) Kaydet: model + test tahminleri + örnek havuzu (Faz 3/4 girdisi)
import pandas as pd
os.makedirs(f"{OUTDIR}/model", exist_ok=True)
trainer.save_model(f"{OUTDIR}/model"); tok.save_pretrained(f"{OUTDIR}/model")

conf = probs.max(1)
df = pd.DataFrame({
    "idx": np.arange(len(labels)), "true": labels, "pred": preds,
    "prob_1": probs[:,1], "confidence": conf, "correct": (preds==labels),
    "text": ds["test"][TEXT_COL], "Haklar": ds["test"]["Haklar"],
})
df.to_csv(f"{OUTDIR}/test_predictions.csv", index=False)

# Nitel analiz havuzu: yüksek-güvenli DOĞRU örnekler, her sınıftan dengeli (~25+25)
pool = (df[df.correct]
        .sort_values("confidence", ascending=False)
        .groupby("true", group_keys=False).head(25))
pool[["idx","true","pred","confidence"]].to_json(
    f"{OUTDIR}/example_pool.json", orient="records", force_ascii=False, indent=2)

json.dump(metrics, open(f"{OUTDIR}/metrics.json","w"), ensure_ascii=False, indent=2)
print("Kaydedildi:", os.listdir(OUTDIR))
print(f"Örnek havuzu: {len(pool)} örnek (sınıf başına ~25).")

## Bitti — indirilecekler (Output sekmesi)
- **`model/`** → Faz 3 XAI için yerelde kullanılacak fine-tuned model (~700 MB).
- **`test_predictions.csv`** → tüm test tahminleri + güven skorları.
- **`example_pool.json`** → nitel ısı haritaları için ~50 yüksek-güvenli doğru örnek.
- **`metrics.json`** → F1-macro, PR-AUC, sınıf-bazlı metrikler.

**Sonraki:** Bu çıktıları `turkish-legal-xai/models/` ve `results/` altına koy; Faz 3'e (attention / rollout / IG / LRP) geçiyoruz.